# Lung Disease Management — Clean Training Pipeline

**Before starting:** Runtime → Change runtime type → T4 GPU

Run each cell in order. Do NOT skip any cell.

In [ ]:
# CELL 1: Check GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU! Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# CELL 2: Clone project
import os, sys, shutil
os.chdir('/content')
if os.path.exists('/content/Lung_Disease_Management'):
    shutil.rmtree('/content/Lung_Disease_Management')
os.system('git clone https://github.com/Sarnika-anbu/Lung_Disease_Management.git /content/Lung_Disease_Management')
os.chdir('/content/Lung_Disease_Management')
sys.path.insert(0, '/content/Lung_Disease_Management')
assert os.path.exists('src/data/preprocess.py'), 'Clone failed!'
print('Clone successful. Working dir:', os.getcwd())

In [ ]:
# CELL 3: Install packages
import os
os.chdir('/content/Lung_Disease_Management')
!pip install -q kaggle timm==0.9.16 grad-cam==1.5.0 librosa==0.10.2 noisereduce==3.0.2
!pip install -q reportlab==4.0.9 pypdf==4.2.0 python-multipart==0.0.9
print('Packages installed')

In [ ]:
# CELL 4: Kaggle credentials
import os, json
os.chdir('/content/Lung_Disease_Management')

KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'  # <-- EDIT THIS
KAGGLE_KEY      = 'YOUR_KAGGLE_API_KEY'   # <-- EDIT THIS

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('Kaggle credentials saved')

In [ ]:
# CELL 5: Download ICBHI dataset
import os
os.chdir('/content/Lung_Disease_Management')
os.makedirs('data/raw/icbhi', exist_ok=True)
!kaggle datasets download -d vbookshelf/respiratory-sound-database -p data/raw/icbhi/ --unzip
!find data/raw/icbhi -name '*.wav' | wc -l

In [ ]:
# CELL 6: Preprocess audio -> spectrograms (30-60 min)
import os
os.chdir('/content/Lung_Disease_Management')
!PYTHONPATH=/content/Lung_Disease_Management python src/data/preprocess.py

In [ ]:
# CELL 7: Verify splits
import os, pandas as pd
os.chdir('/content/Lung_Disease_Management')
train = pd.read_csv('data/processed/splits/train.csv')
test  = pd.read_csv('data/processed/splits/test.csv')
print(f'Train: {len(train)} | Test: {len(test)}')
print(train['disease_class'].value_counts())
assert len(train) > 0, 'ERROR: Train split is empty! Check preprocessing output.'

In [ ]:
# CELL 8: Train model (2-3 hours on T4)
import os
os.chdir('/content/Lung_Disease_Management')
!PYTHONPATH=/content/Lung_Disease_Management python src/training/train.py

In [ ]:
# CELL 9: Evaluate model
import os
os.chdir('/content/Lung_Disease_Management')
!PYTHONPATH=/content/Lung_Disease_Management python src/training/evaluate.py

In [ ]:
# CELL 10: Download checkpoint to your computer
import os
os.chdir('/content/Lung_Disease_Management')
from google.colab import files
if os.path.exists('checkpoints/best.pth'):
    files.download('checkpoints/best.pth')
    print('Downloading best.pth...')
else:
    print('No checkpoint found. Run training first (Cell 8).')